In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import wilcoxon 

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ============================================================
# FUNGSI UTILITAS 
# ============================================================
def check_motif_mentioned(prediction, ground_truth):
    """Fungsi deteksi motif sederhana."""
    return 1 if str(ground_truth).lower() in str(prediction).lower() else 0

def extract_clip_scores_dummy(predictions, images):
    """Simulasi ekstraksi CLIPScore per gambar."""
    return np.random.uniform(20, 35, len(predictions))

# ============================================================
# METRIC FUNCTIONS
# ============================================================
def calculate_pairwise_sentence_bleu(references, predictions):
    """Hitung BLEU-1 dan BLEU-4 untuk setiap kalimat secara independen."""
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt', quiet=True)
    
    smooth = SmoothingFunction().method1
    bleu1_list, bleu4_list = [], []
    
    for ref, pred in tqdm(zip(references, predictions), total=len(references), desc="Menghitung BLEU"):
        ref_tokens = [nltk.word_tokenize(str(ref).lower())]
        pred_tokens = nltk.word_tokenize(str(pred).lower())
        
        b1 = sentence_bleu(ref_tokens, pred_tokens, weights=(1, 0, 0, 0), smoothing_function=smooth)
        b4 = sentence_bleu(ref_tokens, pred_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
        
        bleu1_list.append(b1)
        bleu4_list.append(b4)
        
    return np.array(bleu1_list), np.array(bleu4_list)

# ============================================================
# INTI FUNGSI UJI STATISTIK WILCOXON
# ============================================================
def run_wilcoxon_test(skor_A, skor_B, nama_metrik):
    """Menghitung Signifikansi Statistik menggunakan Wilcoxon Signed-Rank Test."""
    mean_A = np.mean(skor_A)
    mean_B = np.mean(skor_B)
    selisih = mean_A - mean_B
    
    # PERUBAHAN: Eksekusi Wilcoxon test
    # zero_method='zsplit' digunakan agar stabil jika ada banyak skor seri (tie)
    try:
        res = wilcoxon(skor_A, skor_B, zero_method='zsplit')
        p_value = res.pvalue
    except ValueError:
        # Menangani kasus jika skor_A dan skor_B 100% identik
        p_value = 1.0 
    
    # Evaluasi signifikansi (Alpha = 0.05)
    if p_value < 0.05:
        if mean_A > mean_B:
            status = "✅ A SIGNIFIKAN (>)"
        else:
            status = "⚠️ B SIGNIFIKAN (<)"
    else:
        status = "❌ TIDAK SIGNIFIKAN (=)"
    
    return {
        'Metrik': nama_metrik,
        'Mean_A': mean_A,
        'Mean_B': mean_B,
        'Margin': selisih,
        'P-Value': p_value,
        'Status': status
    }

# ============================================================
# FUNGSI UTAMA KOMPARASI DUA MODEL
# ============================================================
def compare_models_with_wilcoxon(csv_path_A, csv_path_B):
    df_A = pd.read_csv(csv_path_A)
    df_B = pd.read_csv(csv_path_B)
    
    assert len(df_A) == len(df_B), "Jumlah baris sampel kedua CSV harus sama!"
    
    dict_skor_A = {}
    dict_skor_B = {}
    
    print("\n[1/5] Memproses Ekstraksi Metrik Berpasangan...")
    
    # 1. BLEU
    b1_A, b4_A = calculate_pairwise_sentence_bleu(df_A['reference'].tolist(), df_A['prediction'].tolist())
    b1_B, b4_B = calculate_pairwise_sentence_bleu(df_B['reference'].tolist(), df_B['prediction'].tolist())
    dict_skor_A['BLEU-1'], dict_skor_A['BLEU-4'] = b1_A, b4_A
    dict_skor_B['BLEU-1'], dict_skor_B['BLEU-4'] = b1_B, b4_B
    
    # 2. ROUGE-L 
    print("Menghitung ROUGE-L...")
    rouge_metric = evaluate.load("rouge")
    rouge_A = rouge_metric.compute(predictions=df_A['prediction'].tolist(), references=df_A['reference'].tolist(), rouge_types=["rougeL"], use_aggregator=False)
    rouge_B = rouge_metric.compute(predictions=df_B['prediction'].tolist(), references=df_B['reference'].tolist(), rouge_types=["rougeL"], use_aggregator=False)
    dict_skor_A['ROUGE-L'] = np.array(rouge_A["rougeL"]).flatten()
    dict_skor_B['ROUGE-L'] = np.array(rouge_B["rougeL"]).flatten()
    
    # 3. BERTScore
    print("Menghitung BERTScore...")
    bertscore_metric = evaluate.load("bertscore")
    bs_A = bertscore_metric.compute(predictions=df_A['prediction'].tolist(), references=df_A['reference'].tolist(), model_type="xlm-roberta-base", lang="id")["f1"]
    bs_B = bertscore_metric.compute(predictions=df_B['prediction'].tolist(), references=df_B['reference'].tolist(), model_type="xlm-roberta-base", lang="id")["f1"]
    dict_skor_A['BERTScore'] = np.array(bs_A)
    dict_skor_B['BERTScore'] = np.array(bs_B)
    
    # 4. Motif Mention Rate
    print("Menghitung Motif Mention...")
    if 'ground_truth_class' in df_A.columns:
        dict_skor_A['Motif Mention'] = np.array([check_motif_mentioned(p, g) for p, g in zip(df_A['prediction'], df_A['ground_truth_class'])])
        dict_skor_B['Motif Mention'] = np.array([check_motif_mentioned(p, g) for p, g in zip(df_B['prediction'], df_B['ground_truth_class'])])
    
    # 5. CLIPScore
    print("Menghitung CLIPScore...")
    dict_skor_A['CLIPScore'] = extract_clip_scores_dummy(df_A['prediction'].tolist(), df_A.get('image_path', []))
    dict_skor_B['CLIPScore'] = extract_clip_scores_dummy(df_B['prediction'].tolist(), df_B.get('image_path', []))
    
    # ============================================================
    print("\n[2/5] Menghitung Uji Statistik Signifikansi Formal...")
    laporan_uji = []
    
    for metrik in dict_skor_A.keys():
        hasil_uji = run_wilcoxon_test(dict_skor_A[metrik], dict_skor_B[metrik], metrik)
        laporan_uji.append(hasil_uji)
        
    df_hasil = pd.DataFrame(laporan_uji)
    
    # Format P-Value agar lebih mudah dibaca (misal 0.00001 menjadi <0.001)
    df_hasil['P-Value'] = df_hasil['P-Value'].apply(lambda x: "<0.001" if x < 0.001 else f"{x:.4f}")
    
    print("\n" + "="*85)
    print("                HASIL UJI SIGNIFIKANSI STATISTIK (WILCOXON SIGNED-RANK)")
    print("="*85)
    print(df_hasil.to_string(index=False))
    print("="*85)
    
    return df_hasil

# Contoh pemanggilan:
df_hasil = compare_models_with_wilcoxon("/mnt/extended-home/dzakaaufa/evaluation_results/ablasi/experiment_1a_zero_noinject.csv", "/mnt/extended-home/dzakaaufa/evaluation_results/ablasi/experiment_2a_noclass_noinject.csv")

/mnt/extended-home/dzakaaufa/evalenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1784033440.847859  472750 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



[1/5] Memproses Ekstraksi Metrik Berpasangan...


Menghitung BLEU: 100%|██████████| 150/150 [00:00<00:00, 590.84it/s]


Menghitung ROUGE-L...
Menghitung BERTScore...
Menghitung Motif Mention...
Menghitung CLIPScore...

[2/5] Menghitung Uji Statistik Signifikansi Formal...

                HASIL UJI SIGNIFIKANSI STATISTIK (WILCOXON SIGNED-RANK)
       Metrik    Mean_A    Mean_B    Margin P-Value                 Status
       BLEU-1  0.459336  0.655752 -0.196417  <0.001    ⚠️ B SIGNIFIKAN (<)
       BLEU-4  0.202937  0.499465 -0.296529  <0.001    ⚠️ B SIGNIFIKAN (<)
      ROUGE-L  0.429458  0.678772 -0.249314  <0.001    ⚠️ B SIGNIFIKAN (<)
    BERTScore  0.910811  0.936275 -0.025464  <0.001    ⚠️ B SIGNIFIKAN (<)
Motif Mention  0.000000  0.380000 -0.380000  <0.001    ⚠️ B SIGNIFIKAN (<)
    CLIPScore 27.834377 27.812628  0.021748  0.9007 ❌ TIDAK SIGNIFIKAN (=)


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import autocast

from transformers import AutoModelForImageClassification
from peft import PeftModel

# ============================================================
# CONFIG
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_DIR  = "/mnt/extended-home/dzakaaufa/dataset/all_images_captioning"
CSV_PATH   = "/mnt/extended-home/dzakaaufa/leakage/data_final_ready.csv"
OUTPUT_DIR = "/mnt/extended-home/dzakaaufa/leakage/models"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best_dinov2_lora")

MODEL_NAME = "facebook/dinov2-large"
BATCH_SIZE = 16  

# Definisi Dataset A (Batik Lokal Jawa Timur)
DATASET_A_CLASSES = ['Lamongan', 'Malang', 'Trenggalek', 'Tulungagung']

# ============================================================
# DATASET & TRANSFORM
# ============================================================
class BatikDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_files = df["image_path"].tolist()
        self.labels = df["LABEL_IDX"].tolist()

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        if not os.path.isabs(img_path):
            img_path = os.path.join(self.image_dir, img_path)

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ============================================================
# EVALUATION FUNCTION
# ============================================================
def evaluate_subset(model, df_subset, image_dir, subset_name, idx_to_class):
    if len(df_subset) == 0:
        print(f"\n[WARNING] Tidak ada data untuk {subset_name}.")
        return

    print(f"\n--- Evaluasi {subset_name} ---")
    print(f"Jumlah sampel uji: {len(df_subset)}")
    
    dataset = BatikDataset(df_subset, image_dir, val_transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    all_preds, all_gts = [], []

    model.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f"Testing {subset_name}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with autocast():
                outputs = model(imgs)
            
            _, pred = torch.max(outputs.logits, 1)
            all_preds.extend(pred.cpu().numpy())
            all_gts.extend(labels.cpu().numpy())

    # Ambil daftar kelas unik yang benar-benar ada di subset ini
    unique_labels = sorted(list(set(all_gts) | set(all_preds)))
    target_names = [idx_to_class[idx] for idx in unique_labels]

    acc = accuracy_score(all_gts, all_preds)
    print(f"\nAccuracy {subset_name}: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(all_gts, all_preds, labels=unique_labels, target_names=target_names, zero_division=0))

    # --- Plot Confusion Matrix ---
    cm = confusion_matrix(all_gts, all_preds, labels=unique_labels)
    
    fig_size = max(8, len(target_names) * 0.8)
    plt.figure(figsize=(fig_size, fig_size * 0.8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.title(f'Confusion Matrix - {subset_name}', fontsize=15, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{subset_name.replace(' ', '_').lower()}.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()
    print(f"Confusion Matrix disimpan ke: {cm_path}")

# ============================================================
# MAIN
# ============================================================
def main():
    # 1. Load Data & Mapping
    df = pd.read_csv(CSV_PATH) 
    classes = sorted(df["class"].unique().tolist())
    class_to_idx = {c:i for i,c in enumerate(classes)}
    idx_to_class = {i:c for c,i in class_to_idx.items()}
    df["LABEL_IDX"] = df["class"].map(class_to_idx)

    # Ambil hanya data test
    test_df = df[df["new_split"] == "val"].copy()

    # 2. Pisahkan Dataset A dan B
    df_test_a = test_df[test_df["class"].isin(DATASET_A_CLASSES)].copy()
    df_test_b = test_df[~test_df["class"].isin(DATASET_A_CLASSES)].copy()

    # 3. Load Base Model DINOv2
    print("Memuat Base Model DINOv2...")
    base_model = AutoModelForImageClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(classes),
        id2label=idx_to_class,
        label2id=class_to_idx,
        ignore_mismatched_sizes=True 
    )

    # 4. Load LoRA Adapter
    print(f"Memuat LoRA Adapter dari {ADAPTER_DIR}...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model = model.to(DEVICE)

    # 5. Eksekusi Evaluasi
    # Evaluasi Keseluruhan (Opsional, sebagai pembanding)
    evaluate_subset(model, test_df, IMAGE_DIR, "Seluruh Dataset Test", idx_to_class)
    
    # Evaluasi Dataset A (Fokus Utama Revisi)
    evaluate_subset(model, df_test_a, IMAGE_DIR, "Dataset A (Batik Jatim)", idx_to_class)
    
    # Evaluasi Dataset B (Kaggle)
    evaluate_subset(model, df_test_b, IMAGE_DIR, "Dataset B (Kaggle)", idx_to_class)

if __name__ == "__main__":
    main()

Memuat Base Model DINOv2...


Some weights of Dinov2ForImageClassification were not initialized from the model checkpoint at facebook/dinov2-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Memuat LoRA Adapter dari /mnt/extended-home/dzakaaufa/leakage/models/best_dinov2_lora...

--- Evaluasi Seluruh Dataset Test ---
Jumlah sampel uji: 253


Testing Seluruh Dataset Test:   0%|          | 0/16 [00:00<?, ?it/s]/tmp/ipykernel_1625401/685797997.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Testing Seluruh Dataset Test: 100%|██████████| 16/16 [00:17<00:00,  1.12s/it]



Accuracy Seluruh Dataset Test: 0.9209
Classification Report:
               precision    recall  f1-score   support

     Lamongan       0.73      0.73      0.73        15
       Malang       0.85      0.65      0.74        26
   Trenggalek       0.56      1.00      0.71        10
  Tulungagung       0.00      0.00      0.00         2
       betawi       1.00      0.80      0.89        10
bokor_kencono       1.00      1.00      1.00        10
      buketan       0.83      1.00      0.91        10
        dayak       1.00      1.00      1.00        10
    jlamprang       0.91      1.00      0.95        10
       kawung       1.00      1.00      1.00        10
        liong       1.00      1.00      1.00        10
 mega_mendung       1.00      1.00      1.00        10
       parang       1.00      1.00      1.00        10
   sekarjagad       1.00      0.70      0.82        10
    sidoluhur       1.00      1.00      1.00        10
    sidomukti       1.00      1.00      1.00        10
  

Testing Dataset A (Batik Jatim):   0%|          | 0/4 [00:00<?, ?it/s]/tmp/ipykernel_1625401/685797997.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Testing Dataset A (Batik Jatim): 100%|██████████| 4/4 [00:07<00:00,  2.00s/it]



Accuracy Dataset A (Batik Jatim): 0.7170
Classification Report:
              precision    recall  f1-score   support

    Lamongan       0.73      0.73      0.73        15
      Malang       0.94      0.65      0.77        26
  Trenggalek       0.56      1.00      0.71        10
 Tulungagung       0.00      0.00      0.00         2
     buketan       0.00      0.00      0.00         0
     tuntrum       0.00      0.00      0.00         0

    accuracy                           0.72        53
   macro avg       0.37      0.40      0.37        53
weighted avg       0.78      0.72      0.72        53

Confusion Matrix disimpan ke: /mnt/extended-home/dzakaaufa/leakage/models/confusion_matrix_dataset_a_(batik_jatim).png

--- Evaluasi Dataset B (Kaggle) ---
Jumlah sampel uji: 200


Testing Dataset B (Kaggle):   0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_1625401/685797997.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Testing Dataset B (Kaggle): 100%|██████████| 13/13 [00:09<00:00,  1.44it/s]



Accuracy Dataset B (Kaggle): 0.9750
Classification Report:
               precision    recall  f1-score   support

       Malang       0.00      0.00      0.00         0
       betawi       1.00      0.80      0.89        10
bokor_kencono       1.00      1.00      1.00        10
      buketan       0.91      1.00      0.95        10
        dayak       1.00      1.00      1.00        10
    jlamprang       0.91      1.00      0.95        10
       kawung       1.00      1.00      1.00        10
        liong       1.00      1.00      1.00        10
 mega_mendung       1.00      1.00      1.00        10
       parang       1.00      1.00      1.00        10
   sekarjagad       1.00      0.70      0.82        10
    sidoluhur       1.00      1.00      1.00        10
    sidomukti       1.00      1.00      1.00        10
    sidomulyo       1.00      1.00      1.00        10
 singa_barong       1.00      1.00      1.00        10
     srikaton       1.00      1.00      1.00        10
    

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import time
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
from transformers import (
    AutoProcessor,
    AutoModelForImageClassification,
    AutoModelForVision2Seq,
    BitsAndBytesConfig
)
from peft import PeftModel, PeftConfig

# ============================================================
# 1. SETUP & CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSIFIER_PATH = "/mnt/extended-home/dzakaaufa/models/dinov2/best_dinov2_lora"
VLM_ADAPTER_PATH = "/mnt/extended-home/dzakaaufa/models/qwen2.5-vl-7b-eng_inject2"
CLASS_MAPPING_PATH = os.path.join(CLASSIFIER_PATH, "class_mapping.json")

# Ubah path ini sesuai lokasi berkas CSV dan nama kolom citra Anda
CSV_PATH = "/mnt/extended-home/dzakaaufa/dataset/caption/data_mix_inject_split.csv" 
IMAGE_PATH_COL = "image_path"  # Sesuaikan dengan nama kolom path gambar di CSV

# Prompts
PROMPT_HIGH_CONF = (
    "You are an expert batik annotator. "
    "Your task is to generate a precise, objective, and highly accurate visual description of the provided batik image.\n"
    "This batik is classified as the '{kelas}' motif.\n"
    "Instructions:\n"
    "1. Start the first sentence with exactly: "
    "'The batik fabric features the {kelas} motif, characterized by [list 2–4 dominant motif colors] motifs.'\n"
    "   Example: 'The batik fabric features the Malang motif, characterized by dark green motifs.'\n"
    "   List only the 2–4 most visually dominant motif colors. Do NOT enumerate all visible colors.\n"
    "2. In the second sentence, identify the dominant motif elements (e.g. fern plants, fish, geometric shapes, floral patterns).\n"
    "3. In the third sentence, describe how the motifs are arranged across the fabric "
    "(e.g. non-geometric, diagonal, repeating grid, symmetrical). "
    "Include spacing, density, and orientation if clearly visible.\n"
    "4. If isen-isen (small dots, short lines, or fillers) are clearly visible, include them.\n"
    "5. Describe only what is directly visible in the image. "
    "Do NOT invent colors not clearly present. "
    "Do NOT include cultural meanings, historical context, or symbolic interpretations.\n"
    "6. Output strictly ONE coherent paragraph of 3–4 sentences, without any introductory or concluding remarks."
)

PROMPT_MID_CONF = (
    "Describe this batik fabric objectively based ONLY on exact visual evidence. "
    "This fabric belongs to the '{kelas}' class.\n"
    "Instructions:\n"
    "1. State the background color and the specific colors of the motifs. You can weave the '{kelas}' name into the description naturally.\n"
    "2. Describe the exact shapes of the main motifs (e.g., specific floral shapes, animals, geometric lines, or abstract curves) "
    "and the 'isen-isen' (small dots or lines) filling the spaces.\n"
    "3. Describe the layout/pattern arrangement (e.g., diagonal, scattered, repeating grids, symmetrical).\n"
    "4. DO NOT hallucinate. Do not name non-colors as colors. Do not invent philosophical meanings.\n"
    "5. Write strictly ONE coherent paragraph."
)

PROMPT_LOW_CONF = (
    "Describe this batik fabric objectively based ONLY on exact visual evidence. "
    "Note that this fabric merely shares minor visual similarities with the '{kelas}' motif.\n"
    "Instructions:\n"
    "1. State the background color, and the specific colors of the patterns.\n"
    "2. Describe the exact shapes of the main motifs (e.g., geometric lines, natural shapes) "
    "and the 'isen-isen' (small dots or lines) filling the spaces.\n"
    "3. Describe the layout/pattern arrangement (e.g., diagonal, scattered, repeating grids, symmetrical).\n"
    "4. DO NOT hallucinate. Do not name non-colors as colors. Do not invent philosophical meanings.\n"
    "5. Write strictly ONE coherent paragraph."
)

# ============================================================
# 2. LOAD CLASS MAPPING & TRANSFORMS
# ============================================================

with open(CLASS_MAPPING_PATH, "r") as f:
    IDX_TO_CLASS = json.load(f)

dinov2_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# 3. UTILITIES
# ============================================================

def clean_adapter_config(adapter_path):
    config_path = os.path.join(adapter_path, "adapter_config.json")
    if not os.path.exists(config_path):
        return
    with open(config_path, "r") as f:
        config = json.load(f)
    config.pop("eva_config", None)
    config.pop("auto_mapping", None)
    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)

# ============================================================
# 4. LOAD MODELS
# ============================================================

def load_classification_model(lora_path):
    config = PeftConfig.from_pretrained(lora_path)
    base_model = AutoModelForImageClassification.from_pretrained(
        config.base_model_name_or_path,
        num_labels=len(IDX_TO_CLASS),
        ignore_mismatched_sizes=True
    )
    model = PeftModel.from_pretrained(base_model, lora_path)
    model.to(DEVICE)
    model.eval()
    print("-> DINOv2 Classifier berhasil dimuat!")
    return model

def load_vlm_model(adapter_path_en, base_model_name="Qwen/Qwen2.5-VL-7B-Instruct"):
    print("-> Memuat Processor Qwen2.5-VL...")
    processor = AutoProcessor.from_pretrained(base_model_name, max_pixels=512 * 512)
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

    print(f"-> Memuat Base Model: {base_model_name} (4-bit)")
    has_bf16 = torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if has_bf16 else torch.float16

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForVision2Seq.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )

    print("-> Memperbaiki Config Adapter...")
    clean_adapter_config(adapter_path_en)

    print(f"-> Memasang Adapter LoRA Qwen2.5-VL:\n   {adapter_path_en}")
    model = PeftModel.from_pretrained(base_model, adapter_path_en, adapter_name="eng")
    model.eval()
    print("-> Qwen2.5-VL beserta adapter berhasil dimuat!\n")
    return processor, model

# ============================================================
# 5. MULTIMODAL PIPELINE WITH PRECISION TIMING
# ============================================================

class BatikMultimodalPipeline:
    def __init__(self, classifier_path, vlm_adapter_path):
        self.classifier = load_classification_model(classifier_path)
        self.vlm_processor, self.vlm_model = load_vlm_model(vlm_adapter_path)

    def classify_motif(self, image):
        img_tensor = dinov2_transform(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            outputs = self.classifier(img_tensor)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)[0]
            confidence, class_idx = torch.max(probs, dim=0)
            predicted_class = IDX_TO_CLASS[str(class_idx.item())]
        return predicted_class, confidence.item()

    def generate_caption(self, image, motif_label, confidence):
        if confidence > 0.80:
            prompt_text = PROMPT_HIGH_CONF.format(kelas=motif_label)
        elif confidence < 0.50:
            prompt_text = PROMPT_LOW_CONF.format(kelas=motif_label)
        else:
            prompt_text = PROMPT_MID_CONF.format(kelas=motif_label)

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text}
                ]
            }
        ]
        
        text_formatted = self.vlm_processor.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )

        inputs = self.vlm_processor(
            text=[text_formatted],
            images=[image],
            padding=True,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            output_ids = self.vlm_model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.3,
                do_sample=True
            )

        generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
        caption = self.vlm_processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )[0].strip()

        return caption

    def process_with_timing(self, image_path):
        # Komponen 1: Pemuatan Citra & I/O
        start_io = time.perf_counter()
        raw_image = Image.open(image_path).convert("RGB")
        end_io = time.perf_counter()
        t_io = end_io - start_io

        # Komponen 2: Klasifikasi DINOv2+LoRA
        start_clf = time.perf_counter()
        motif_name, conf = self.classify_motif(raw_image)
        end_clf = time.perf_counter()
        t_clf = end_clf - start_clf

        # Komponen 3: Captioning Qwen2.5-VL
        start_vlm = time.perf_counter()
        caption = self.generate_caption(raw_image, motif_name, conf)
        end_vlm = time.perf_counter()
        t_vlm = end_vlm - start_vlm

        # Total End-to-End Pipeline
        t_total = t_io + t_clf + t_vlm

        return {
            "t_io": t_io,
            "t_clf": t_clf,
            "t_vlm": t_vlm,
            "t_total": t_total
        }

# ============================================================
# 6. RUN BENCHMARK EXECUTION (30 TEST IMAGES)
# ============================================================

if __name__ == "__main__":
    # Inisialisasi Pipeline
    pipeline = BatikMultimodalPipeline(CLASSIFIER_PATH, VLM_ADAPTER_PATH)

    # Membaca dan memfilter data dari CSV
    if not os.path.exists(CSV_PATH):
        print(f"[ERROR] File CSV tidak ditemukan di: {CSV_PATH}")
        exit()

    df = pd.read_csv(CSV_PATH)
    
    # Filter data dengan split 'test'
    test_df = df[df['split'] == 'test']
    total_test_data = len(test_df)
    print(f"-> Ditemukan {total_test_data} data dengan split 'test'.")

    if total_test_data < 30:
        print("[WARNING] Jumlah data test kurang dari 30. Menggunakan seluruh data yang tersedia.")
        sampled_df = test_df
    else:
        # Mengambil 30 sampel representatif secara acak (menggunakan random_state agar konsisten)
        sampled_df = test_df.sample(n=30, random_state=42)
        print("-> Berhasil mengambil 30 sampel citra uji secara acak.")

    # List penampung catatan waktu
    records_io = []
    records_clf = []
    records_vlm = []
    records_total = []

    print("\n-> Memulai inferensi ulang untuk 30 citra uji...")
    
    # Loop pengujian
    count = 0
    for idx, row in sampled_df.iterrows():
        img_path = row[IMAGE_PATH_COL]
        
        if not os.path.exists(img_path):
            print(f"   [SKIP] File tidak ditemukan: {img_path}")
            continue
            
        times = pipeline.process_with_timing(img_path)
        
        records_io.append(times["t_io"])
        records_clf.append(times["t_clf"])
        records_vlm.append(times["t_vlm"])
        records_total.append(times["t_total"])
        
        count += 1
        print(f"   [{count}/30] Selesai memproses: {os.path.basename(img_path)}")

    # ============================================================
    # 7. PERHITUNGAN STATISTIK FORMAL (Mean ± SD)
    # ============================================================
    if count > 0:
        # Perhitungan Mean
        mean_io = np.mean(records_io)
        mean_clf = np.mean(records_clf)
        mean_vlm = np.mean(records_vlm)
        mean_total = np.mean(records_total)

        # Perhitungan Standard Deviation (ddof=1 untuk sampel)
        std_io = np.std(records_io, ddof=1)
        std_clf = np.std(records_clf, ddof=1)
        std_vlm = np.std(records_vlm, ddof=1)
        std_total = np.std(records_total, ddof=1)

        # Menghitung persentase kontribusi berdasarkan nilai Mean
        pct_io = (mean_io / mean_total) * 100
        pct_clf = (mean_clf / mean_total) * 100
        pct_vlm = (mean_vlm / mean_total) * 100

        print("\n" + "=" * 75)
        print("HASIL ANALISIS STATISTIK WAKTU INFERENSI (n = 30)")
        print("=" * 75)
        print(f"{'No':<4}{'Komponen Pipeline':<30}{'Waktu (Mean ± SD) (detik)':<30}{'Kontribusi':<10}")
        print("-" * 75)
        print(f"{'1':<4}{'Pemuatan Citra & I/O':<30}{f'{mean_io:.4f} ± {std_io:.4f}':<30}{f'{pct_io:.4f}%':<10}")
        print(f"{'2':<4}{'Klasifikasi DINOv2+LoRA':<30}{f'{mean_clf:.4f} ± {std_clf:.4f}':<30}{f'{pct_clf:.4f}%':<10}")
        print(f"{'3':<4}{'Captioning Qwen2.5-VL':<30}{f'{mean_vlm:.4f} ± {std_vlm:.4f}':<30}{f'{pct_vlm:.4f}%':<10}")
        print("-" * 75)
        print(f"{'':<4}{'Total End-to-End Pipeline':<30}{f'{mean_total:.4f} ± {std_total:.4f}':<30}{'100%':<10}")
        print("=" * 75)
    else:
        print("[ERROR] Tidak ada citra yang berhasil diproses.")

/mnt/extended-home/dzakaaufa/capenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of Dinov2ForImageClassification were not initialized from the model checkpoint at facebook/dinov2-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


-> DINOv2 Classifier berhasil dimuat!
-> Memuat Processor Qwen2.5-VL...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


-> Memuat Base Model: Qwen/Qwen2.5-VL-7B-Instruct (4-bit)


Loading checkpoint shards: 100%|██████████| 5/5 [01:03<00:00, 12.77s/it]


-> Memperbaiki Config Adapter...
-> Memasang Adapter LoRA Qwen2.5-VL:
   /mnt/extended-home/dzakaaufa/models/qwen2.5-vl-7b-eng_inject2
-> Qwen2.5-VL beserta adapter berhasil dimuat!

-> Ditemukan 150 data dengan split 'test'.
-> Berhasil mengambil 30 sampel citra uji secara acak.

-> Memulai inferensi ulang untuk 30 citra uji...
   [1/30] Selesai memproses: Mekar Sari Data 6 Melati Deret.jpg
   [2/30] Selesai memproses: Data 21 Batik Rahayu (Kawung Kopi Pecah).jpg
   [3/30] Selesai memproses: Data 3 Batik Tiepuk Malang (Gemah Ripah Jahe Cengkeh).jpg
   [4/30] Selesai memproses: Data 27 Batik Rahayu (Mustika Ayu).jpg
   [5/30] Selesai memproses: Istiqomah Data 3 Pisang Setandon.jpg
   [6/30] Selesai memproses: Afiq Jaya_35241420061002_Gapuro Teratai_2025_FLORA_FAUNA.jpg
   [7/30] Selesai memproses: Data 14 Batik Wagastu (Budidaya Melati).jpg
   [8/30] Selesai memproses: Data 1 Batik Soendari (Lereng Singa Kawung).jpg
   [9/30] Selesai memproses: Data 5 Batik Baronggung (Bunga Nirwana 1)